In [1]:
from sqlalchemy import String, Numeric, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase): pass

class Customer(Base):
    __tablename__ = "customers"
    id:      Mapped[str] = mapped_column(String, primary_key=True)
    name:    Mapped[str] = mapped_column(String)
    segment: Mapped[str] = mapped_column(String, default="retail")     # retail, HNI, NRI

class Account(Base):
    __tablename__ = "accounts"
    number:      Mapped[str]   = mapped_column(String, primary_key=True)
    customer_id: Mapped[str]   = mapped_column(ForeignKey("customers.id"))
    acc_type:    Mapped[str]   = mapped_column(String)                 # savings, current, salary
    balance:     Mapped[float] = mapped_column(Numeric(18, 2))

class Transaction(Base):
    __tablename__ = "transactions"
    id:             Mapped[int]   = mapped_column(primary_key=True)
    account_number: Mapped[str]   = mapped_column(ForeignKey("accounts.number"))
    txn_type:       Mapped[str]   = mapped_column(String)              # credit, debit
    amount:         Mapped[float] = mapped_column(Numeric(18, 2))
    description:    Mapped[str]   = mapped_column(String)

class Loan(Base):
    __tablename__ = "loans"
    number:      Mapped[str]   = mapped_column(String, primary_key=True)
    customer_id: Mapped[str]   = mapped_column(ForeignKey("customers.id"))
    loan_type:   Mapped[str]   = mapped_column(String)                 # home, personal
    outstanding: Mapped[float] = mapped_column(Numeric(18, 2))
    emi:         Mapped[float] = mapped_column(Numeric(18, 2))

class Card(Base):
    __tablename__ = "cards"
    number:      Mapped[str] = mapped_column(String, primary_key=True)
    customer_id: Mapped[str] = mapped_column(ForeignKey("customers.id"))
    card_type:   Mapped[str] = mapped_column(String)                   # debit, credit
    status:      Mapped[str] = mapped_column(String, default="active")

In [2]:
import os
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

engine  = create_engine(os.getenv("MCP_DATABASE_URL", "sqlite:///banking.db"),
                        connect_args={"check_same_thread": False})
Session = sessionmaker(bind=engine)
Base.metadata.create_all(engine)        # creates all 5 tables (no-op if they exist)

In [3]:
from sqlalchemy import select

def seed():
    s = Session()
    if not s.scalar(select(Customer)):
        s.add_all([
            Customer(id="c1", name="Amit Sharma", segment="HNI"),
            Customer(id="c2", name="Priya Nair",  segment="retail"),
            Account(number="ACC-0001", customer_id="c1", acc_type="savings", balance=250000),
            Account(number="ACC-0002", customer_id="c1", acc_type="current", balance=80000),
            Account(number="ACC-0003", customer_id="c2", acc_type="salary",  balance=84000),
            Transaction(account_number="ACC-0001", txn_type="credit", amount=120000, description="Salary"),
            Transaction(account_number="ACC-0001", txn_type="debit",  amount=35000,  description="Rent"),
            Transaction(account_number="ACC-0003", txn_type="credit", amount=84000,  description="Salary"),
            Loan(number="LOAN-001", customer_id="c1", loan_type="home", outstanding=3200000, emi=42000),
            Card(number="XXXX-1234", customer_id="c1", card_type="credit", status="active"),
            Card(number="XXXX-5678", customer_id="c2", card_type="debit",  status="active"),
        ])
        s.commit()
    s.close()

seed()

In [4]:
from sqlalchemy import select

def _customer(s, name):
    return s.scalar(select(Customer).where(Customer.name.ilike(f"%{name}%")))

def get_accounts(name) -> dict:
    s = Session(); c = _customer(s, name)
    if not c: s.close(); return {"error": f"not found: {name}"}
    rows = s.scalars(select(Account).where(Account.customer_id == c.id)).all(); s.close()
    return {"customer": c.name,
            "accounts": [{"number": a.number, "type": a.acc_type, "balance": float(a.balance)} for a in rows]}

def get_transactions(account_number) -> dict:
    s = Session()
    rows = s.scalars(select(Transaction).where(Transaction.account_number == account_number)).all(); s.close()
    return {"account": account_number,
            "transactions": [{"type": t.txn_type, "amount": float(t.amount), "desc": t.description} for t in rows]}

def get_loans(name) -> dict:
    s = Session(); c = _customer(s, name)
    if not c: s.close(); return {"error": f"not found: {name}"}
    rows = s.scalars(select(Loan).where(Loan.customer_id == c.id)).all(); s.close()
    return {"customer": c.name,
            "loans": [{"number": l.number, "type": l.loan_type,
                       "outstanding": float(l.outstanding), "emi": float(l.emi)} for l in rows]}

def get_cards(name) -> dict:
    s = Session(); c = _customer(s, name)
    if not c: s.close(); return {"error": f"not found: {name}"}
    rows = s.scalars(select(Card).where(Card.customer_id == c.id)).all(); s.close()
    return {"customer": c.name,
            "cards": [{"number": cd.number, "type": cd.card_type, "status": cd.status} for cd in rows]}

def get_customer_360(name) -> dict:
    a = get_accounts(name)
    if "error" in a: return a
    return {**a, "loans": get_loans(name)["loans"], "cards": get_cards(name)["cards"]}

In [5]:
import json
from fastmcp import FastMCP

mcp = FastMCP("Banking Server",
              instructions="Accounts, transactions, loans, cards, and a customer 360 view.")

@mcp.tool()
def tool_get_accounts(customer_name: str) -> str:
    """List a customer's accounts. Example: Amit Sharma"""
    return json.dumps(get_accounts(customer_name), default=str)

@mcp.tool()
def tool_get_transactions(account_number: str) -> str:
    """List an account's transactions. Example: ACC-0001"""
    return json.dumps(get_transactions(account_number), default=str)

@mcp.tool()
def tool_get_loans(customer_name: str) -> str:
    """List a customer's loans. Example: Amit Sharma"""
    return json.dumps(get_loans(customer_name), default=str)

@mcp.tool()
def tool_get_cards(customer_name: str) -> str:
    """List a customer's cards. Example: Amit Sharma"""
    return json.dumps(get_cards(customer_name), default=str)

@mcp.tool()
def tool_get_customer_360(customer_name: str) -> str:
    """Full 360 view: accounts + loans + cards. Example: Amit Sharma"""
    return json.dumps(get_customer_360(customer_name), default=str)

In [6]:
import asyncio, nest_asyncio
nest_asyncio.apply()
loop = asyncio.get_event_loop()

print("registered:", [t.name for t in loop.run_until_complete(mcp.list_tools())])

res = loop.run_until_complete(
    mcp.call_tool("tool_get_customer_360", {"customer_name": "Amit Sharma"}))
print(res.content[0].text)        # the JSON an AI client receives

registered: ['tool_get_accounts', 'tool_get_transactions', 'tool_get_loans', 'tool_get_cards', 'tool_get_customer_360']
{"customer": "Amit Sharma", "accounts": [{"number": "ACC-0001", "type": "savings", "balance": 250000.0}, {"number": "ACC-0002", "type": "current", "balance": 80000.0}], "loans": [{"number": "LOAN-001", "type": "home", "outstanding": 3200000.0, "emi": 42000.0}], "cards": [{"number": "XXXX-1234", "type": "credit", "status": "active"}]}


In [ ]:
seed()
await mcp.run_async(transport="streamable-http")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 4.1.0                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      Banking Server, 4.1.0                       │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[10/08/26 22:27:12] INFO     Starting MCP server 'Banking Server' with transport 'streamable-http' ]8;id=11709166;file:///Users/bnbaliga/Projects/Python/hands-on-agentic-ai/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=11709167;file:///Users/bnbaliga/Projects/Python/hands-on-agentic-ai/.venv/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#358\358]8;;\
                             on http://127.0.0.1:8000/mcp                                                          

INFO:     Started server process [26482]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
